# Chapter 15: factor models and attribution

The default uses 120 constructed months from January 2010, five normal factor columns, RF=0.001 and one asset generated with seed 20261015. FF3 and FF5 share the same constructed factor columns here. Fit CAPM, FF3 and FF5; inspect HAC sensitivity, variance decomposition and attribution; then save coefficient CSV, summary JSON and new figures under RUN_DIR. The historical French snapshots and the project's nine-model report have separate identities.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch15_factor_models_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

import pandas as pd


from quant_course.factor_models import fit_monthly_factor_model
from ch15_factor_models import (
    build_summary,
    fit_models,
    load_sample,
    make_figures,
    write_artifacts,
)
from IPython.display import display


## 1. Constructed sample and sources

Historical loader functions are imported but not called. The executable cell constructs one asset and shared factor arrays; it neither downloads market data nor validates a historical snapshot or fills missing months.

In [ ]:
import numpy as np
rng = np.random.default_rng(20261015)
idx = pd.period_range("2010-01", periods=120, freq="M")
factors = pd.DataFrame(rng.normal(0, .025, (120, 5)), index=idx, columns=["Mkt-RF", "SMB", "HML", "RMW", "CMA"])
factors["RF"] = .001
frames = {"ff3": factors[["Mkt-RF", "SMB", "HML", "RF"]].copy(), "ff5": factors.copy()}
total = pd.Series(.001 + .002 + factors["Mkt-RF"] + .3*factors["HML"] + rng.normal(0, .01, 120), index=idx, name="CONSTRUCTED asset")
print("CONSTRUCTED seeded factor-regression inputs; French branch skipped")
assert len(total) == 120 and frames["ff5"].index.equals(total.index)


Inputs are decimal monthly returns: 0.01 means 1% per month. PeriodIndex labels intervals, not availability or execution times. Historical FF3 and FF5 releases need not have identical Mkt-RF or SMB columns; this constructed example deliberately shares them.

In [ ]:
display(total.head().to_frame())
display(frames["ff3"].head())
display(frames["ff5"].head())

## 2. OLS factor exposures

CAPM uses Mkt-RF; FF3 adds SMB and HML; FF5 adds RMW and CMA. Regress the asset's excess return on the selected factors, subtracting matching RF once. Compare the shared CAPM core coefficient table with the chapter workflow, then inspect all three models.

In [ ]:
core_capm = fit_monthly_factor_model(
    total,
    frames["ff3"],
    factor_columns=("Mkt-RF",),
    hac_maxlags=6,
)
fits = fit_models(total, frames, hac_maxlags=6)

assert core_capm.coefficient_table.equals(fits["capm"].coefficient_table)
print("共享核心和章节工作流的 CAPM 系数表一致。")
print("模型：", list(fits))

## 3. Monthly intercept uncertainty

Compare Bartlett HAC lags 3, 6 and 12, measured in monthly observations. The intercept point estimate stays fixed while standard errors and 95% intervals change. Finite-sample correction and a t reference distribution give approximate inference under conditions; output alone does not verify those conditions or tradability.

In [ ]:
summary = build_summary(total, frames, fits, constructed=True)

alpha_rows = []
for name in ("capm", "ff3", "ff5"):
    model = summary["models"][name]
    for lag, values in model["alpha_hac_lag_sensitivity"].items():
        alpha_rows.append({
            "model": name,
            "HAC lags (months)": int(lag),
            "alpha (% per month)": values["estimate"] * 100,
            "HAC standard error (% per month)": values["standard_error_hac"] * 100,
            "95% CI low (% per month)": values["ci_low_hac"] * 100,
            "95% CI high (% per month)": values["ci_high_hac"] * 100,
        })

alpha_sensitivity = pd.DataFrame(alpha_rows)
display(alpha_sensitivity.pivot(
    index="model",
    columns="HAC lags (months)",
    values="alpha (% per month)",
))

These outputs use the fixed constructed sample. Historical chapter intercept estimates are not results of this computation.

In [ ]:
alpha_intervals = alpha_sensitivity[
    ["model", "HAC lags (months)", "HAC standard error (% per month)",
     "95% CI low (% per month)", "95% CI high (% per month)"]
]
display(alpha_intervals)

## 4. In-sample variance decomposition

With an intercept, fitted values and residuals are sample-orthogonal. Check Var(y)=Var(fitted)+Var(residual), using n−1 throughout. The fitted-sample identity does not establish future residual independence or prediction.

In [ ]:
variance_rows = []
for name in ("capm", "ff3", "ff5"):
    decomposition = summary["models"][name]["sample_variance_decomposition_ddof1"]
    variance_rows.append({
        "model": name,
        "observed excess variance, ddof=1": decomposition["observed_excess_return"],
        "fitted excess variance, ddof=1": decomposition["fitted_excess_return"],
        "residual variance, ddof=1": decomposition["residual"],
        "fitted + residual": decomposition["sum_fitted_plus_residual"],
        "identity verified": decomposition["identity_holds_within_tolerance"],
    })

display(pd.DataFrame(variance_rows).set_index("model"))

## 5. FF5 exposures and figures

Loadings are dimensionless slopes on the constructed factors, with HAC intervals from the shared core. Mkt-RF is already excess. The new plots describe this fit.

In [ ]:
display(fits["ff5"].coefficient_table)
print("R²:", fits["ff5"].r_squared)
print("Adjusted R²:", fits["ff5"].adjusted_r_squared)

The contribution plot adds monthly factor, intercept and residual terms arithmetically, multiplying by 100 for percentage points. It is neither NAV nor compounded wealth. A regression contribution is not automatically an investable zero-dollar portfolio.

In [ ]:
from IPython.display import Image, display

figure_info = make_figures(total, frames, fits, output_dir=RUN_DIR)
for key, metadata in figure_info.items():
    image_path = ROOT / metadata["path"]
    print(key, metadata["width_px"], "x", metadata["height_px"], "px")
    display(Image(filename=str(image_path)))

## 6. Outputs and interpretation

The final cell writes coefficients.csv and summary.json under RUN_DIR. Historical French FF3/FF5 common-name factor differences have unresolved upstream causes, so those regressions are not strictly nested and their R² difference cannot all be attributed to RMW/CMA. That historical warning does not describe the shared constructed arrays above. No stock PIT panel, fills, transaction costs or net alpha is supplied.

In [ ]:
written = write_artifacts(fits, summary, coefficients_path=RUN_DIR / "coefficients.csv", summary_path=RUN_DIR / "summary.json")
print(written)
